# The 42 טף occurrences in the Septuagint


## Table of contents (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Introduction</a>
* <a href="#bullet2">2 - Install the required libraries</a>
* <a href="#bullet3">3 - Load the BHSA occurrence list</a>
* <a href="#bullet4">4 - Load Perseus MCP</a>
* <a href="#bullet5">5 - Handle MCP responses</a>
* <a href="#bullet6">6 - Map the Hebrew books to LXX editions</a>
* <a href="#bullet7">7 - Explain the Hebrew-to-Greek mapping</a>
* <a href="#bullet8">8 - Build the 42 candidate passage URNs</a>
* <a href="#bullet9">9 - Retrieve the Greek passages</a>
* <a href="#bullet10">10 - Seek likely Greek representatives of טף</a>
* <a href="#bullet11">11 - Inspect the proposed alignments</a>
* <a href="#bullet12">12 - Keep a review queue</a>
* <a href="#bullet13">13 - Export results and provenance</a>
* <a href="#bullet14">14 - Final checks</a>
* <a href="#bullet15">15 - Required libraries</a>
* <a href="#bullet16">16 - Responsible disclosure</a>
* <a href="#bullet17">17 - Licences</a>
* <a href="#bullet18">18 - Notebook details</a>

# 1 - Introduction<a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

This Jupyter Notebook starts with the 42 occurrences of **טף** (*ṭap̄*) retrieved from the local BHSA 2021 corpus by the Context-Fabric notebook and then seeks the corresponding Septuagint passages through Perseus MCP. SHEBANQ is not an input here; its public records serve only as a separate control on the Hebrew occurrence set.

A passage match and a lexical alignment are different judgments. A Hebrew reference can often be carried directly to the Greek book, but that does not yet tell us which Greek word represents טף. Some books also differ in numbering or arrangement. The notebook therefore records the reference mapping and the proposed lexical mapping separately, and it leaves doubtful rows visible rather than forcing a one-to-one answer.

The question we would like to answer is:

> At each of the 42 BHSA occurrences of טף, which LXX passage corresponds to the Hebrew verse, and which Greek expression is the most plausible representative of טף?

The procedure has three stages:

1. import the BHSA references and their Hebrew context;
2. map each Hebrew book and verse to a CTS passage in the selected LXX edition; and
3. mark Greek words belonging to the listed candidate families.

The third stage proposes candidates for review; it is not a dictionary lookup. Greek `ἀποσκευή`, for example, can refer to baggage or dependants, while `παιδία`, `νήπια`, and `ἔκγονα` describe people in different ways. Elsewhere the translation may recast the group as a household (`οἶκος` or `πανοικία`) or use the unusual collective `ἀπαρτία`. Where several candidates occur, where no candidate is found, or where the reference systems diverge, the row is sent to a review queue.

**Input**

* `context_fabric/results/taf_occurrences.csv`
* `context_fabric/results/audit.json`

**Main output**

* `lxx/taf_lxx_results/alignment.csv`
* `lxx/taf_lxx_results/review_queue.csv`
* `lxx/taf_lxx_results/alignment_mapping_provenance.json`

# 2 - Install the required libraries<a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

The local `perseus-mcp` package is opened through an in-process FastMCP client. Passage text still comes from the Perseus/Scaife services through MCP tools.

Restart the kernel after installation if the imports fail in an already-running environment.

In [1]:
# Install the notebook dependencies, including MCP 1.30.0.
%pip install -q -r "requirements.txt"


Note: you may need to restart the kernel to use updated packages.


# 3 - Load the BHSA occurrence list<a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

The input is not a fresh LXX search. It is the 42-row table exported by the Context-Fabric notebook in this repository. Context-Fabric supplied the BHSA references, verse and word nodes, matched Hebrew forms, and complete verse text. The `context_frame` column contains researcher-supplied classifications based on close reading; it is separate from the BHSA features retrieved by Context-Fabric.

The LXX alignment does not depend on accepting those contextual labels. They are retained as useful descriptive metadata, while the Hebrew-to-Greek reference and lexical alignments are assessed separately. The accompanying audit records the checks made by the same Context-Fabric run, including the total of 42 occurrences.

Each run has a unique ID and a UTC start time recorded before source retrieval.
Each MCP request records its own start and end time, including failures.
The final export records completion time, retrieval dates, software versions, editions, and service endpoints.
These dates describe this session, including any cached responses; they are not publication dates.

The recorded run uses Tony Jurg, *Perseus MCP*, version 1.0.2 (Zenodo, 2026), [doi:10.5281/zenodo.20931139](https://doi.org/10.5281/zenodo.20931139).


In [2]:
# Load the BHSA occurrence list
from __future__ import annotations

import importlib.metadata as package_metadata
import json
import logging
import re
import unicodedata
from pathlib import Path
from typing import Any

import pandas as pd


def find_notebook_dir() -> Path:
    here = Path.cwd().resolve()
    if (here / "taf_lxx_alignment.ipynb").exists():
        return here
    if (here / "lxx" / "taf_lxx_alignment.ipynb").exists():
        return here / "lxx"
    raise FileNotFoundError("Start Jupyter in the repository root or in the lxx directory.")


NOTEBOOK_DIR = find_notebook_dir()
REPOSITORY_DIR = NOTEBOOK_DIR.parent
INPUT_CSV = REPOSITORY_DIR / "context_fabric" / "results" / "taf_occurrences.csv"
AUDIT_JSON = REPOSITORY_DIR / "context_fabric" / "results" / "audit.json"
OUTPUT_DIR = NOTEBOOK_DIR / "taf_lxx_results"

occurrences = pd.read_csv(INPUT_CSV)
audit = json.loads(AUDIT_JSON.read_text(encoding="utf-8"))

required_columns = {
    "reference",
    "verse_node",
    "word_node",
    "matched_form",
    "verse_text",
    "context_frame",
}
assert required_columns <= set(occurrences.columns)
assert len(occurrences) == 42
assert occurrences["reference"].nunique() == 42
assert audit["taf_total"] == 42

display(occurrences.head())
print(f"Imported {len(occurrences)} BHSA occurrences from {INPUT_CSV.relative_to(REPOSITORY_DIR)}")

# Start recording before any source retrieval.
import sys
if str(NOTEBOOK_DIR.parent) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR.parent))
from notebook_provenance import RunProvenance
from perseus_mcp import server
run_provenance = RunProvenance("lxx_alignment", OUTPUT_DIR, server)


,reference,verse_node,word_node,matched_form,verse_text,context_frame,context_detail,relevant_hebrew_phrase,taph_rendering,working_translation,classification_note
0,Genesis 34:29,1415398,18987,טַפָּם֙,וְאֶת־כָּל־חֵילָ֤ם וְאֶת־כָּל־טַפָּם֙ וְאֶת־נְ...,"conflict, captivity, or lethal judgment",capture and plunder after the massacre at Shechem,ואת־כל־טפם ואת־נשיהם שבו,their little ones,and they took captive all their little ones an...,"The finite verb שבו, 'they took captive', dire..."
1,Genesis 43:8,1415687,24293,טַפֵּֽנוּ׃,וַיֹּ֨אמֶר יְהוּדָ֜ה אֶל־יִשְׂרָאֵ֣ל אָבִ֗יו ש...,household provision or survival,a journey undertaken so that the household may...,ונחיה ולא נמות גם־אנחנו גם־אתה גם־טפנו,our little ones,"so that we may live and not die—we, you, and o...","Travel is proposed, but the clause itself clim..."
2,Genesis 45:19,1415766,25818,טַפְּכֶם֙,וְאַתָּ֥ה צֻוֵּ֖יתָה זֹ֣את עֲשׂ֑וּ קְחוּ־לָכֶם...,"household movement, displacement, or protected...",state-assisted household migration to Egypt,קחו־לכם מארץ מצרים עגלות לטפכם ולנשיכם,your little ones,take wagons from the land of Egypt for your li...,The wagons are expressly provided to transport...
3,Genesis 46:5,1415780,26076,טַפָּם֙,וַיָּ֥קָם יַעֲקֹ֖ב מִבְּאֵ֣ר שָׁ֑בַע וַיִּשְׂא...,"household movement, displacement, or protected...",household migration from Canaan to Egypt,וישאו בני־ישראל את־יעקב אביהם ואת־טפם ואת־נשיה...,their little ones,"the sons of Israel carried Jacob their father,...",The little ones are named among those physical...
4,Genesis 47:12,1415821,26806,טָּֽף׃,וַיְכַלְכֵּ֤ל יֹוסֵף֙ אֶת־אָבִ֣יו וְאֶת־אֶחָ֔י...,household provision or survival,food allocation according to household need,ויכלכל יוסף את־אביו ואת־אחיו ואת כל־בית אביו ל...,the household dependants,"Joseph provided his father, his brothers, and ...",לפי הטף is literally 'according to the mouth o...


Imported 42 BHSA occurrences from context_fabric\results\taf_occurrences.csv


# 4 - Load Perseus MCP<a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

We inspect the live tool registry before retrieving passages. This catches a changed or incomplete MCP installation early, rather than halfway through the 42 rows.

In [3]:
# Load Perseus MCP
from fastmcp import Client
from perseus_mcp import server

mcp = server.mcp
logging.getLogger("fastmcp").setLevel(logging.CRITICAL)

async with Client(mcp) as client:
    tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in tools}
required_tools = {
    "get_author_resources",
    "get_scaife_library_metadata",
    "get_scaife_passage_text",
}
missing = required_tools - set(tool_by_name)
assert not missing, f"Required Perseus MCP tools are missing: {sorted(missing)}"

print(f"Loaded MCP server: {mcp.name}")
print(f"Perseus MCP version: {package_metadata.version('perseus-mcp')}")

Loaded MCP server: perseus
Perseus MCP version: 1.0.2


# 5 - Handle MCP responses<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

A failed request is recorded in the alignment table, so one unavailable verse does not discard the other results.

In [4]:
# Handle MCP responses
def tool_text(result: Any) -> str:
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_text(client: Client, tool_name: str, arguments: dict[str, Any] | None = None) -> str:
    result = await run_provenance.call_tool(client, tool_name, arguments or {})
    return tool_text(result)


async def call_json(client: Client, tool_name: str, arguments: dict[str, Any] | None = None) -> Any:
    return json.loads(await call_text(client, tool_name, arguments))


def clean_text(value: str) -> str:
    return " ".join(str(value or "").split())


def normalise_greek(value: str) -> str:
    decomposed = unicodedata.normalize("NFD", str(value).lower())
    return "".join(character for character in decomposed if unicodedata.category(character) != "Mn")


def greek_words(value: str) -> list[str]:
    return re.findall(r"[^\W\d_]+", str(value), flags=re.UNICODE)

# 6 - Map the Hebrew books to LXX editions<a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

The 42 references span twelve Hebrew books. Each book is assigned an explicit Scaife work and edition URN; the Greek catalogue, for example, lists Ezra under **2 Esdras**.

The notebook checks work labels against the CTS catalogue when it is available and verifies all twelve editions through Scaife metadata. If the CTS catalogue is unavailable, the provenance records the fallback to Scaife.
These edition checks are separate from passage retrieval.


In [5]:
# Map the Hebrew books to LXX editions
LXX_TEXTGROUP = "urn:cts:greekLit:tlg0527"

BOOK_MAP = {
    "Genesis": {"lxx_title": "Genesis", "work": "tlg001"},
    "Exodus": {"lxx_title": "Exodus", "work": "tlg002"},
    "Numbers": {"lxx_title": "Numbers", "work": "tlg004"},
    "Deuteronomy": {"lxx_title": "Deuteronomy", "work": "tlg005"},
    "Joshua": {"lxx_title": "Joshua", "work": "tlg006"},
    "Judges": {"lxx_title": "Judges", "work": "tlg008"},
    "2 Samuel": {"lxx_title": "2 Samuel", "work": "tlg012"},
    "Jeremiah": {"lxx_title": "Jeremiah", "work": "tlg049"},
    "Ezekiel": {"lxx_title": "Ezekiel", "work": "tlg053"},
    "Esther": {"lxx_title": "Esther", "work": "tlg019"},
    "Ezra": {"lxx_title": "2 Esdras", "work": "tlg018"},
    "2 Chronicles": {"lxx_title": "2 Chronicles", "work": "tlg016"},
}

for item in BOOK_MAP.values():
    item["work_urn"] = f"{LXX_TEXTGROUP}.{item['work']}"
    item["edition_urn"] = f"{item['work_urn']}.1st1K-grc1"

catalogue_lookup = {"tool": "get_author_resources", "status": "ok"}
try:
    async with Client(mcp) as client:
        catalogue = await call_json(
            client,
            "get_author_resources",
            {"author": LXX_TEXTGROUP, "language": "greek"},
        )
except Exception as exc:
    catalogue_lookup = {
        "tool": "get_author_resources",
        "status": "unavailable",
        "error": f"{type(exc).__name__}: {exc}",
        "fallback": "Validate every mapped work and edition through Scaife metadata below.",
    }
    print("CTS catalogue unavailable; checking every edition through Scaife metadata.")
else:
    catalogue_works = {
        title: work["urn"]
        for author in catalogue.get("authors", [])
        for work in author.get("works", [])
        for title in work.get("titles", [])
    }
    
    for hebrew_book, item in BOOK_MAP.items():
        assert catalogue_works[item["lxx_title"]] == item["work_urn"], hebrew_book

edition_checks = {}
async with Client(mcp) as client:
    for hebrew_book, item in BOOK_MAP.items():
        metadata = await call_json(
            client,
            "get_scaife_library_metadata",
            {"urn": item["work_urn"]},
        )
        available_editions = {text["urn"] for text in metadata.get("texts", [])}
        edition_checks[hebrew_book] = item["edition_urn"] in available_editions

assert all(edition_checks.values()), edition_checks

book_map_df = pd.DataFrame(
    [
        {"BHSA book": book, **item}
        for book, item in BOOK_MAP.items()
    ]
)
book_map_df[["BHSA book", "lxx_title", "edition_urn"]]

,BHSA book,lxx_title,edition_urn
0,Genesis,Genesis,urn:cts:greekLit:tlg0527.tlg001.1st1K-grc1
1,Exodus,Exodus,urn:cts:greekLit:tlg0527.tlg002.1st1K-grc1
2,Numbers,Numbers,urn:cts:greekLit:tlg0527.tlg004.1st1K-grc1
3,Deuteronomy,Deuteronomy,urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1
4,Joshua,Joshua,urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1
5,Judges,Judges,urn:cts:greekLit:tlg0527.tlg008.1st1K-grc1
6,2 Samuel,2 Samuel,urn:cts:greekLit:tlg0527.tlg012.1st1K-grc1
7,Jeremiah,Jeremiah,urn:cts:greekLit:tlg0527.tlg049.1st1K-grc1
8,Ezekiel,Ezekiel,urn:cts:greekLit:tlg0527.tlg053.1st1K-grc1
9,Esther,Esther,urn:cts:greekLit:tlg0527.tlg019.1st1K-grc1


# 7 - Explain the Hebrew-to-Greek mapping<a class="anchor" id="bullet7"></a>
##### [Back to ToC](#TOC)

There are three different mappings in the output:

| Layer | From | To | What it establishes |
|---|---|---|---|
| book mapping | BHSA book name | Scaife LXX edition URN | which Greek work is queried |
| reference mapping | BHSA chapter and verse | LXX passage locator | which Greek passage is the candidate parallel |
| lexical proposal | the טף occurrence in its Hebrew verse | one or more Greek tokens in that passage | which Greek wording deserves inspection |

Most rows begin with the same chapter and verse in both traditions. Two known exceptions in this dataset are entered explicitly: Hebrew Deut 29:10 points to the audience wording in Greek 29.11, and Hebrew Josh 8:35 points to Greek Josh 9.8.

Jeremiah receives a warning because the Greek and Masoretic arrangements diverge substantially. A same-number Jeremiah verse may be retrievable while still being the wrong textual passage. Such rows should remain in review state until their reference override has been checked from a verse map or the surrounding content.

In [6]:
# Record known reference overrides and caution flags
REFERENCE_OVERRIDES = {
    "Deuteronomy 29:10": {
        "lxx_locator": "29.11",
        "reason": "The women, ἔκγονα, and resident alien occur in Greek 29.11.",
    },
    "Joshua 8:35": {
        "lxx_locator": "9.8",
        "reason": "The covenant-reading audience occurs in Greek Joshua 9.8.",
    },
}

BOOKS_REQUIRING_REFERENCE_REVIEW = {
    "Jeremiah": "LXX Jeremiah differs from the Masoretic order after chapter 25.",
}

# Candidate families are broad enough to seek plausible renderings but small
# enough to inspect. Stems are compared after accents and breathings are removed.
GREEK_CANDIDATE_FAMILIES = {
    "ἀποσκευή": ["αποσκευ"],
    "ἔκγονον": ["εκγον"],
    "παιδίον / παῖς": ["παιδ", "παισ"],
    "τέκνον": ["τεκν"],
    "νήπιος": ["νηπι"],
    "βρέφος": ["βρεφ"],
    "σῶμα": ["σωμα"],
    "οἶκος / πανοικία": ["οικ", "πανοικ"],
    "ἀπαρτία": ["απαρτι"],
    "ἐπίγονος / ἐπιγονή": ["επιγον"],
}

reference_rules = pd.DataFrame(
    [
        {"BHSA reference": reference, **details}
        for reference, details in REFERENCE_OVERRIDES.items()
    ]
)
display(reference_rules)

,BHSA reference,lxx_locator,reason
0,Deuteronomy 29:10,29.11,"The women, ἔκγονα, and resident alien occur in..."
1,Joshua 8:35,9.8,The covenant-reading audience occurs in Greek ...


# 8 - Build the 42 candidate passage URNs<a class="anchor" id="bullet8"></a>
##### [Back to ToC](#TOC)

The BHSA references use underscores in numbered book names. We retain the original reference and add parsed columns beside it. The resulting CTS passage URN is therefore auditable row by row.

In [7]:
# Build the 42 candidate passage URNs
REFERENCE_PATTERN = re.compile(r"^(?P<book>.+) (?P<chapter>\d+):(?P<verse>\d+)$")


def parse_bhsa_reference(reference: str) -> tuple[str, int, int]:
    match = REFERENCE_PATTERN.fullmatch(reference)
    if not match:
        raise ValueError(f"Could not parse BHSA reference: {reference}")
    book = match.group("book").replace("_", " ")
    return book, int(match.group("chapter")), int(match.group("verse"))


alignment = occurrences.copy()
alignment[["book", "chapter", "verse"]] = alignment["reference"].apply(
    lambda reference: pd.Series(parse_bhsa_reference(reference))
)

assert set(alignment["book"]) == set(BOOK_MAP)

alignment["lxx_title"] = alignment["book"].map(lambda book: BOOK_MAP[book]["lxx_title"])
alignment["lxx_edition_urn"] = alignment["book"].map(lambda book: BOOK_MAP[book]["edition_urn"])
alignment["direct_locator"] = alignment.apply(lambda row: f"{row.chapter}.{row.verse}", axis=1)
alignment["lxx_locator"] = alignment.apply(
    lambda row: REFERENCE_OVERRIDES.get(row.reference, {}).get("lxx_locator", row.direct_locator),
    axis=1,
)
alignment["reference_method"] = alignment["reference"].map(
    lambda reference: "curated override" if reference in REFERENCE_OVERRIDES else "same chapter and verse"
)
alignment["reference_note"] = alignment.apply(
    lambda row: REFERENCE_OVERRIDES.get(row.reference, {}).get(
        "reason", BOOKS_REQUIRING_REFERENCE_REVIEW.get(row.book, "")
    ),
    axis=1,
)
alignment["lxx_passage_urn"] = alignment.apply(
    lambda row: f"{row.lxx_edition_urn}:{row.lxx_locator}", axis=1
)

assert len(alignment) == 42
assert alignment["lxx_passage_urn"].nunique() == 42

alignment[["reference", "lxx_title", "lxx_locator", "reference_method", "reference_note"]]

,reference,lxx_title,lxx_locator,reference_method,reference_note
0,Genesis 34:29,Genesis,34.29,same chapter and verse,
1,Genesis 43:8,Genesis,43.8,same chapter and verse,
2,Genesis 45:19,Genesis,45.19,same chapter and verse,
3,Genesis 46:5,Genesis,46.5,same chapter and verse,
4,Genesis 47:12,Genesis,47.12,same chapter and verse,
5,Genesis 47:24,Genesis,47.24,same chapter and verse,
6,Genesis 50:8,Genesis,50.8,same chapter and verse,
7,Genesis 50:21,Genesis,50.21,same chapter and verse,
8,Exodus 10:10,Exodus,10.10,same chapter and verse,
9,Exodus 10:24,Exodus,10.24,same chapter and verse,


# 9 - Retrieve the Greek passages<a class="anchor" id="bullet9"></a>
##### [Back to ToC](#TOC)

Each passage is requested separately through `get_scaife_passage_text`. If Scaife is temporarily unavailable or a locator does not exist, the exception is recorded in `retrieval_error`. The exported table keeps retrieval failures visible alongside the passages that were returned.

In [8]:
# Retrieve the Greek passages
passage_texts: dict[str, str] = {}
retrieval_errors: dict[str, str] = {}

async with Client(mcp) as client:
    for number, urn in enumerate(alignment["lxx_passage_urn"], start=1):
        try:
            passage_texts[urn] = clean_text(
                await call_text(client, "get_scaife_passage_text", {"urn": urn})
            )
        except Exception as exc:
            passage_texts[urn] = ""
            retrieval_errors[urn] = f"{type(exc).__name__}: {exc}"
        if number % 10 == 0 or number == len(alignment):
            print(f"Retrieved or attempted {number:>2} of {len(alignment)} passages")

alignment["greek_text"] = alignment["lxx_passage_urn"].map(passage_texts)
alignment["retrieval_error"] = alignment["lxx_passage_urn"].map(retrieval_errors).fillna("")

print(f"Passages with text: {(alignment['greek_text'] != '').sum()}")
print(f"Retrieval errors: {(alignment['retrieval_error'] != '').sum()}")

Retrieved or attempted 10 of 42 passages


Retrieved or attempted 20 of 42 passages


Retrieved or attempted 30 of 42 passages


Retrieved or attempted 40 of 42 passages


Retrieved or attempted 42 of 42 passages
Passages with text: 42
Retrieval errors: 0


# 10 - Seek likely Greek representatives of טף<a class="anchor" id="bullet10"></a>
##### [Back to ToC](#TOC)

The following function does not decide the translation. It merely finds tokens belonging to the candidate families listed above. The exact Greek token is preserved, while the family label makes summary counts possible.

Genesis 34:29, for example, contains both `σώματα` and `ἀποσκευήν`. Elsewhere, `παῖδες` may denote servants rather than small children. These possibilities require comparison of the Hebrew and Greek clauses.

In [9]:
# Seek likely Greek representatives of טף
def candidate_terms(text: str) -> list[dict[str, str]]:
    found: list[dict[str, str]] = []
    seen: set[tuple[str, str]] = set()
    for token in greek_words(text):
        normalised = normalise_greek(token)
        for family, stems in GREEK_CANDIDATE_FAMILIES.items():
            if any(normalised.startswith(stem) for stem in stems):
                key = (family, token)
                if key not in seen:
                    found.append({"family": family, "token": token})
                    seen.add(key)
    return found


alignment["candidate_details"] = alignment["greek_text"].map(candidate_terms)
alignment["candidate_families"] = alignment["candidate_details"].map(
    lambda rows: " | ".join(dict.fromkeys(row["family"] for row in rows))
)
alignment["candidate_greek_tokens"] = alignment["candidate_details"].map(
    lambda rows: " | ".join(dict.fromkeys(row["token"] for row in rows))
)
alignment["candidate_family_count"] = alignment["candidate_details"].map(
    lambda rows: len({row["family"] for row in rows})
)


def alignment_status(row: pd.Series) -> str:
    if row.retrieval_error:
        return "retrieval failed"
    if row.book in BOOKS_REQUIRING_REFERENCE_REVIEW and row.reference not in REFERENCE_OVERRIDES:
        return "reference review required"
    if row.candidate_family_count == 0:
        return "no lexical candidate found"
    if row.candidate_family_count > 1:
        return "multiple lexical candidates"
    if row.reference_method == "curated override":
        return "candidate found after reference override"
    return "single lexical candidate"


alignment["alignment_status"] = alignment.apply(alignment_status, axis=1)
alignment[[
    "reference",
    "lxx_locator",
    "candidate_families",
    "candidate_greek_tokens",
    "alignment_status",
]]

,reference,lxx_locator,candidate_families,candidate_greek_tokens,alignment_status
0,Genesis 34:29,34.29,σῶμα | ἀποσκευή | οἶκος / πανοικία,σώματα | ἀποσκευὴν | οἰκίαις,multiple lexical candidates
1,Genesis 43:8,43.8,παιδίον / παῖς | ἀποσκευή,παιδάριον | ἀποσκευὴ,multiple lexical candidates
2,Genesis 45:19,45.19,παιδίον / παῖς,παιδίοις,single lexical candidate
3,Genesis 46:5,46.5,ἀποσκευή,ἀποσκευὴν,single lexical candidate
4,Genesis 47:12,47.12,οἶκος / πανοικία | σῶμα,οἴκῳ | σῶμα,multiple lexical candidates
5,Genesis 47:24,47.24,οἶκος / πανοικία,οἴκοις,single lexical candidate
6,Genesis 50:8,50.8,οἶκος / πανοικία,πανοικία | οἰκία,single lexical candidate
7,Genesis 50:21,50.21,οἶκος / πανοικία,οἰκίας,single lexical candidate
8,Exodus 10:10,10.10,ἀποσκευή,ἀποσκευὴν,single lexical candidate
9,Exodus 10:24,10.24,ἀποσκευή,ἀποσκευὴ,single lexical candidate


# 11 - Inspect the proposed alignments<a class="anchor" id="bullet11"></a>
##### [Back to ToC](#TOC)

The summary answers two useful preliminary questions: how many references yielded Greek text, and which candidate families occur most often. It should not be read as a translation-frequency table until the review queue has been worked through.

In [10]:
# Inspect the proposed alignments
status_summary = (
    alignment.groupby("alignment_status", dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["rows", "alignment_status"], ascending=[False, True])
)

family_summary = (
    alignment.assign(candidate_family=alignment["candidate_families"].str.split(" | ", regex=False))
    .explode("candidate_family")
    .query("candidate_family != ''")
    .groupby("candidate_family")
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["rows", "candidate_family"], ascending=[False, True])
)

book_summary = (
    alignment.groupby(["book", "alignment_status"])
    .size()
    .rename("rows")
    .reset_index()
)

display(status_summary)
display(family_summary)
display(book_summary)

,alignment_status,rows
4,single lexical candidate,27
1,multiple lexical candidates,9
3,reference review required,3
2,no lexical candidate found,2
0,candidate found after reference override,1


,candidate_family,rows
6,ἀποσκευή,13
1,οἶκος / πανοικία,11
2,παιδίον / παῖς,11
4,τέκνον,6
5,ἀπαρτία,4
3,σῶμα,2
8,ἔκγονον,2
0,νήπιος,1
7,ἐπίγονος / ἐπιγονή,1


,book,alignment_status,rows
0,2 Chronicles,single lexical candidate,2
1,2 Samuel,single lexical candidate,1
2,Deuteronomy,multiple lexical candidates,2
3,Deuteronomy,single lexical candidate,5
4,Esther,no lexical candidate found,2
5,Exodus,single lexical candidate,3
6,Ezekiel,multiple lexical candidates,1
7,Ezra,single lexical candidate,1
8,Genesis,multiple lexical candidates,3
9,Genesis,single lexical candidate,5


# 12 - Keep a review queue<a class="anchor" id="bullet12"></a>
##### [Back to ToC](#TOC)

A row leaves the automatic path when the passage could not be retrieved, the book has a known reference-order problem, no candidate family was found, or more than one family was found. The queue retains both texts for comparison of their wording and context.

In [11]:
# Keep a review queue
review_queue = alignment.loc[
    alignment["alignment_status"] != "single lexical candidate",
    [
        "reference",
        "matched_form",
        "verse_text",
        "context_frame",
        "lxx_passage_urn",
        "reference_method",
        "reference_note",
        "greek_text",
        "candidate_families",
        "candidate_greek_tokens",
        "alignment_status",
        "retrieval_error",
    ],
].copy()

print(f"Rows requiring inspection: {len(review_queue)}")
review_queue

Rows requiring inspection: 15


,reference,matched_form,verse_text,context_frame,lxx_passage_urn,reference_method,reference_note,greek_text,candidate_families,candidate_greek_tokens,alignment_status,retrieval_error
0,Genesis 34:29,טַפָּם֙,וְאֶת־כָּל־חֵילָ֤ם וְאֶת־כָּל־טַפָּם֙ וְאֶת־נְ...,"conflict, captivity, or lethal judgment",urn:cts:greekLit:tlg0527.tlg001.1st1K-grc1:34.29,same chapter and verse,,καὶ πάντα τὰ σώματα καὶ πᾶσαν τὴν ἀποσκευὴν αὐ...,σῶμα | ἀποσκευή | οἶκος / πανοικία,σώματα | ἀποσκευὴν | οἰκίαις,multiple lexical candidates,
1,Genesis 43:8,טַפֵּֽנוּ׃,וַיֹּ֨אמֶר יְהוּדָ֜ה אֶל־יִשְׂרָאֵ֣ל אָבִ֗יו ש...,household provision or survival,urn:cts:greekLit:tlg0527.tlg001.1st1K-grc1:43.8,same chapter and verse,,εἶπεν δὲ Ἰούδας πρὸς Ἰσραὴλ τὸν πατέρα αὐτοῦ Ἀ...,παιδίον / παῖς | ἀποσκευή,παιδάριον | ἀποσκευὴ,multiple lexical candidates,
4,Genesis 47:12,טָּֽף׃,וַיְכַלְכֵּ֤ל יֹוסֵף֙ אֶת־אָבִ֣יו וְאֶת־אֶחָ֔י...,household provision or survival,urn:cts:greekLit:tlg0527.tlg001.1st1K-grc1:47.12,same chapter and verse,,καὶ ἐσιτομέτρει Ἰωσὴφ τῷ πατρὶ αὐτοῦ καὶ τοῖς ...,οἶκος / πανοικία | σῶμα,οἴκῳ | σῶμα,multiple lexical candidates,
13,Numbers 16:27,טַפָּֽם׃,וַיֵּעָל֗וּ מֵעַ֧ל מִשְׁכַּן־קֹ֛רֶח דָּתָ֥ן וַ...,"conflict, captivity, or lethal judgment",urn:cts:greekLit:tlg0527.tlg004.1st1K-grc1:16.27,same chapter and verse,,καὶ ἀπέστησαν ἀπὸ τῆς 13 εἰς γην] pr εις| γῆς ...,ἀπαρτία | τέκνον | ἀποσκευή,ἀπαρτία | τέκνα | ἀποσκευὴ,multiple lexical candidates,
17,Numbers 32:16,טַפֵּֽנוּ׃,וַיִּגְּשׁ֤וּ אֵלָיו֙ וַ֣יֹּאמְר֔וּ גִּדְרֹ֥ת ...,"household movement, displacement, or protected...",urn:cts:greekLit:tlg0527.tlg004.1st1K-grc1:32.16,same chapter and verse,,καὶ προσῆλθον αὐτῷ καὶ ἔλεγον᾿Επαύλεις προβάτω...,οἶκος / πανοικία | ἀποσκευή,οἰκοδομήσωμεν | ἀποσκευαῖς,multiple lexical candidates,
19,Numbers 32:24,טַפְּכֶ֔ם,בְּנֽוּ־לָכֶ֤ם עָרִים֙ לְטַפְּכֶ֔ם וּגְדֵרֹ֖ת ...,"household movement, displacement, or protected...",urn:cts:greekLit:tlg0527.tlg004.1st1K-grc1:32.24,same chapter and verse,,καὶ οἰκοδομήσητε ὑμῖν αὐτοῖς πόλεις τῇ ἀποσκευ...,οἶκος / πανοικία | ἀποσκευή,οἰκοδομήσητε | ἀποσκευῇ,multiple lexical candidates,
25,Deuteronomy 20:14,טַּ֨ף,רַ֣ק הַ֠נָּשִׁים וְהַטַּ֨ף וְהַבְּהֵמָ֜ה וְכֹל...,"conflict, captivity, or lethal judgment",urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:20.14,same chapter and verse,,πλὴν τῶν γυναικῶν καὶ τῆς ἀποσκευῆς· καὶ ὅσα ἂ...,ἀποσκευή | ἀπαρτία,ἀποσκευῆς | ἀπαρτίαν,multiple lexical candidates,
26,Deuteronomy 29:10,טַפְּכֶ֣ם,טַפְּכֶ֣ם נְשֵׁיכֶ֔ם וְגֵ֣רְךָ֔ אֲשֶׁ֖ר בְּקֶ֣...,"covenant, cultic, or administrative inclusion",urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:29.11,curated override,"The women, ἔκγονα, and resident alien occur in...",αἱ γυναῖκες ὑμῶν καὶ τὰ ἔκγονα ὑμῶν καὶ ὁ προσ...,ἔκγονον | τέκνον,ἔκγονα | τέκνα,multiple lexical candidates,
29,Joshua 8:35,טַּ֔ף,לֹֽא־הָיָ֣ה דָבָ֔ר מִכֹּ֖ל אֲשֶׁר־צִוָּ֣ה מֹשׁ...,"covenant, cultic, or administrative inclusion",urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8,curated override,The covenant-reading audience occurs in Greek ...,οὐκ ἦν ἀπὸ πόντων ὧν ἐνετείλατο Μωυσῆς τῷ Ἰησο...,παιδίον / παῖς,παιδίοις,candidate found after reference override,
33,Jeremiah 40:7,טָ֔ף,וַיִּשְׁמְעוּ֩ כָל־שָׂרֵ֨י הַחֲיָלִ֜ים אֲשֶׁ֣ר...,"household movement, displacement, or protected...",urn:cts:greekLit:tlg0527.tlg049.1st1K-grc1:40.7,same chapter and verse,LXX Jeremiah differs from the Masoretic order ...,καὶ ἐπιστρέψω τὴν ἀποικίαν Ἰούδα καὶ τὴν ἀποικ...,οἶκος / πανοικία,οἰκοδομήσω,reference review required,


# 13 - Export results and provenance<a class="anchor" id="bullet13"></a>
##### [Back to ToC](#TOC)

The principal CSV retains every BHSA column, the CTS mapping, the Greek passage, the candidate terms, and the status. A second CSV contains only the review queue. JSON files preserve the mapping rules and run metadata so that later manual corrections can be distinguished from the automatic first pass.

In [12]:
# Export results and provenance
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_alignment = alignment.drop(columns=["candidate_details"])
export_alignment.to_csv(OUTPUT_DIR / "alignment.csv", index=False, encoding="utf-8")
review_queue.to_csv(OUTPUT_DIR / "review_queue.csv", index=False, encoding="utf-8")
status_summary.to_csv(OUTPUT_DIR / "status_summary.csv", index=False, encoding="utf-8")
family_summary.to_csv(OUTPUT_DIR / "candidate_summary.csv", index=False, encoding="utf-8")

mapping_provenance = {
    **run_provenance.finish([item["edition_urn"] for item in BOOK_MAP.values()]),
    "input_csv": str(INPUT_CSV.relative_to(REPOSITORY_DIR)),
    "input_audit": str(AUDIT_JSON.relative_to(REPOSITORY_DIR)),
    "bhsa_rows": len(occurrences),
    "lxx_textgroup": LXX_TEXTGROUP,
    "book_map": BOOK_MAP,
    "catalogue_lookup": catalogue_lookup,
    "reference_overrides": REFERENCE_OVERRIDES,
    "books_requiring_reference_review": BOOKS_REQUIRING_REFERENCE_REVIEW,
    "greek_candidate_families": GREEK_CANDIDATE_FAMILIES,
    "perseus_mcp_version": package_metadata.version("perseus-mcp"),
}
(OUTPUT_DIR / "alignment_mapping_provenance.json").write_text(
    json.dumps(mapping_provenance, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Wrote alignment results to {OUTPUT_DIR.relative_to(REPOSITORY_DIR).as_posix()}")

Wrote alignment results to lxx/taf_lxx_results


# 14 - Final checks<a class="anchor" id="bullet14"></a>
##### [Back to ToC](#TOC)

The final assertions concern the reproducible mechanics: 42 BHSA rows remain 42 output rows, every book has a mapping, every row has a CTS candidate URN, and every row receives a status.

In [13]:
# Final checks
assert len(alignment) == 42
assert alignment["reference"].nunique() == 42
assert alignment["lxx_edition_urn"].notna().all()
assert alignment["lxx_passage_urn"].str.startswith("urn:cts:greekLit:tlg0527.").all()
assert alignment["alignment_status"].notna().all()
assert set(review_queue.index) <= set(alignment.index)

print("All BHSA-to-LXX alignment checks passed.")

All BHSA-to-LXX alignment checks passed.


# 15 - Required libraries<a class="anchor" id="bullet15"></a>
##### [Back to ToC](#TOC)

The notebook uses:

* `pandas` for the 42-row table and CSV exports;
* `fastmcp` for the in-process client; and
* `perseus-mcp` for the live catalogue and passage tools.

Network access is required for the Perseus/Scaife calls. The input BHSA files are local to this repository.

# 16 - Responsible disclosure<a class="anchor" id="bullet16"></a>
##### [Back to ToC](#TOC)

The author used OpenAI Codex to assist with Python code and data analysis. The author reviewed and edited the material and takes responsibility for the content and interpretation.

# 17 - Licences<a class="anchor" id="bullet17"></a>
##### [Back to ToC](#TOC)

The original code in this notebook is licensed under the [MIT License](../LICENSE).
The original research writing and explanatory text are licensed under [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
Credit Tony Jurg, link the licence and repository, and indicate any changes when reusing the original text; see the [text licence](../LICENSE-TEXT.md).

Quoted source material, retrieved passages, and datasets retain their own terms.
See [Sources, attribution, and reuse](../DATA_RIGHTS.md) for their licences and attribution requirements.


# 18 - Notebook details<a class="anchor" id="bullet18"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr><td><strong>Author</strong></td><td>Tony Jurg</td></tr>
    <tr><td><strong>Version</strong></td><td>1.6</td></tr>
    <tr><td><strong>Date</strong></td><td>October 26, 2026</td></tr>
  </table>
</div>
